# MVCC and the WAL — step by step

A hands-on lab. Run one cell at a time, read what happened, then move on. You will query system columns (xmin, xmax, ctid), watch row versions change after an UPDATE, observe dead tuples accumulate, and run VACUUM to reclaim space.

Each step shows the **plain SQL** too, so you can copy it and run it in psql, the Lakebase SQL editor, or any tool. The Python just runs that SQL against your branch.

**Before you start:** finish `00-setup/` so `LB_PROFILE`, `LB_PROJECT`, `LB_BRANCH`, `LB_SCHEMA` are set and the factory is loaded on your branch.

## Setup — connect to your branch

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..', 'common', 'scripts'))
from lakebase import connect, run_sql

SCHEMA = os.environ.get('LB_SCHEMA', 'factory')
print('connected; schema:', SCHEMA)

## Step 1 — Query system columns

Postgres stores hidden columns on every row: xmin (transaction that inserted), xmax (transaction that deleted, or 0 if live), ctid (physical location). The SQL:

```sql
SELECT xmin, xmax, ctid, machine_id, health_pct
  FROM factory.machine
 WHERE machine_id = %s
```

In [ ]:
machines = run_sql("SELECT machine_id FROM factory.machine LIMIT 1")
if not machines:
    print("No machines found")
else:
    machine_id = machines[0][0]
    sql = """
    SELECT xmin, xmax, ctid, machine_id, health_pct
      FROM factory.machine
     WHERE machine_id = %s
    """
    with connect() as conn:
        with conn.cursor() as cur:
            cur.execute(sql, (machine_id,))
            row = cur.fetchone()
            if row:
                print(f"xmin={row[0]}, xmax={row[1]}, ctid={row[2]}")
                print(f"machine_id={row[3]}, health_pct={row[4]}")

## Step 2 — UPDATE and watch versioning

Run an UPDATE and see xmin, xmax, and ctid change as Postgres creates a new version. The SQL:

```sql
SELECT xmin, xmax, ctid, health_pct FROM factory.machine WHERE machine_id = %s;
UPDATE factory.machine SET health_pct = 50.0 WHERE machine_id = %s;
SELECT xmin, xmax, ctid, health_pct FROM factory.machine WHERE machine_id = %s
```

In [ ]:
machines = run_sql("SELECT machine_id FROM factory.machine LIMIT 1")
if not machines:
    print("No machines found")
else:
    machine_id = machines[0][0]

    with connect() as conn:
        with conn.cursor() as cur:
            # Query before
            cur.execute("SELECT xmin, xmax, ctid, health_pct FROM factory.machine WHERE machine_id = %s", (machine_id,))
            before = cur.fetchone()
            print("Before UPDATE:")
            if before:
                print(f"  xmin={before[0]}, xmax={before[1]}, ctid={before[2]}, health_pct={before[3]}")

            # UPDATE
            cur.execute("UPDATE factory.machine SET health_pct = 50.0 WHERE machine_id = %s", (machine_id,))
            print("\nRunning UPDATE...")

            # Query after
            cur.execute("SELECT xmin, xmax, ctid, health_pct FROM factory.machine WHERE machine_id = %s", (machine_id,))
            after = cur.fetchone()
            print("\nAfter UPDATE:")
            if after:
                print(f"  xmin={after[0]}, xmax={after[1]}, ctid={after[2]}, health_pct={after[3]}")

            if before and after:
                if before[0] != after[0]:
                    print(f"\nNote: xmin changed ({before[0]} → {after[0]}), new version created")
                if before[2] != after[2]:
                    print(f"Note: ctid changed ({before[2]} → {after[2]}), new physical location")

## Step 3 — Dead tuples accumulate

Run multiple UPDATEs and watch the dead tuple count grow. The SQL:

```sql
SELECT n_live_tup, n_dead_tup FROM pg_stat_user_tables WHERE relname = 'machine';
-- Run multiple UPDATEs
UPDATE factory.machine SET health_pct = health_pct + 1.0 WHERE machine_id = %s  -- repeat 10 times
```

In [ ]:
machines = run_sql("SELECT machine_id FROM factory.machine LIMIT 1")
if not machines:
    print("No machines found")
else:
    machine_id = machines[0][0]

    with connect() as conn:
        with conn.cursor() as cur:
            # Before UPDATEs
            cur.execute("SELECT n_live_tup, n_dead_tup FROM pg_stat_user_tables WHERE relname = 'machine'")
            before = cur.fetchone()
            print("Before UPDATEs:")
            if before:
                print(f"  n_live_tup={before[0]}, n_dead_tup={before[1]}")

            # Run 10 UPDATEs
            for i in range(10):
                cur.execute("UPDATE factory.machine SET health_pct = health_pct + 1.0 WHERE machine_id = %s", (machine_id,))

            print("\nAfter 10 UPDATEs:")

            # After UPDATEs
            cur.execute("SELECT n_live_tup, n_dead_tup FROM pg_stat_user_tables WHERE relname = 'machine'")
            after = cur.fetchone()
            if after:
                print(f"  n_live_tup={after[0]}, n_dead_tup={after[1]}")
                if before and after[1] > before[1]:
                    print(f"  Dead tuples increased by {after[1] - before[1]}")

## Step 4 — VACUUM reclaims space

Run VACUUM to remove dead tuples and reclaim space. The SQL:

```sql
SELECT n_live_tup, n_dead_tup FROM pg_stat_user_tables WHERE relname = 'machine';
VACUUM factory.machine;
SELECT n_live_tup, n_dead_tup FROM pg_stat_user_tables WHERE relname = 'machine'
```

In [ ]:
with connect() as conn:
    with conn.cursor() as cur:
        # Before VACUUM
        cur.execute("SELECT n_live_tup, n_dead_tup FROM pg_stat_user_tables WHERE relname = 'machine'")
        before = cur.fetchone()
        print("Before VACUUM:")
        if before:
            print(f"  n_live_tup={before[0]}, n_dead_tup={before[1]}")

        # VACUUM
        cur.execute("VACUUM factory.machine")
        print("\nRunning VACUUM...")

        # After VACUUM
        cur.execute("SELECT n_live_tup, n_dead_tup FROM pg_stat_user_tables WHERE relname = 'machine'")
        after = cur.fetchone()
        print("\nAfter VACUUM:")
        if after:
            print(f"  n_live_tup={after[0]}, n_dead_tup={after[1]}")
            if before and after[1] < before[1]:
                print(f"  Dead tuples reclaimed: {before[1] - after[1]}")

## What you learned

You explored MVCC: Postgres stores multiple versions of rows (xmin, xmax, ctid) so readers do not block writers. UPDATEs create new versions, leaving old ones marked dead. VACUUM removes dead tuples and reclaims space. This is the machinery behind "consistent reads" and "no blocking."

You have mastered the core SQL, performance, and concurrency features of Postgres. From window functions and recursive CTEs to JSONB, spatial queries, search, RLS, constraints, indexes, and MVCC, Postgres gives you a complete data engine.